# 04_glove_fasttext: Character N-Grams and Out-of-Vocabulary Resolution

This notebook trains a FastText model using Gensim to show how character n-grams resolve Out-of-Vocabulary (OOV) lookup failures that cause static Word2Vec to crash.


In [1]:
from gensim.models import FastText
from gensim.models import Word2Vec

# Tiny training corpus
sentences = [
    ["the", "cat", "sat", "on", "the", "mat"],
    ["feline", "sat", "on", "the", "rug"],
    ["dogs", "run", "in", "the", "garden"]
]

# 1. Train standard Word2Vec (vocabulary is fixed to training tokens)
w2v = Word2Vec(sentences, vector_size=10, window=2, min_count=1, epochs=10)

# 2. Train FastText (stores character n-grams)
ft = FastText(sentences, vector_size=10, window=2, min_count=1, min_n=3, max_n=6, epochs=10)

# 3. Attempt OOV word retrieval (e.g. 'cats' - not in training vocabulary)
print("Vocabulary keys in training data:", list(w2v.wv.key_to_index.keys()))

try:
    vector = w2v.wv["cats"]
except KeyError as e:
    print("\n[Word2Vec Error]: Word 'cats' is out of vocabulary!")

# FastText handles the OOV word via character subword n-grams
ft_vector = ft.wv["cats"]
print("\nFastText Vector for OOV word 'cats':\n", ft_vector)

# Similar words lookup
print("\nFastText similarity 'cat' vs 'cats':", ft.wv.similarity("cat", "cats"))


Vocabulary keys in training data: ['the', 'on', 'sat', 'garden', 'in', 'run', 'dogs', 'rug', 'feline', 'mat', 'cat']

[Word2Vec Error]: Word 'cats' is out of vocabulary!

FastText Vector for OOV word 'cats':
 [ 3.6268285e-03  2.6440994e-05 -2.2043910e-02  1.3001269e-02
 -1.0199426e-02 -1.7043855e-02 -1.0019243e-03 -1.8407776e-05
  1.5739180e-02  1.2835404e-02]

FastText similarity 'cat' vs 'cats': 0.161751


### Output Explanation
- Standard Word2Vec raises a `KeyError` when queried with the Out-of-Vocabulary word `"cats"`.
- FastText handles this by decomposing `"cats"` into character n-grams (e.g., `cat`, `ats`) and summing their vectors to generate an embedding.
